## Import library

In [1]:
import yaml
import os
import boto3
import pandas as pd
from pprint import pprint
import shutil
import pickle
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelBinarizer

import warnings
warnings.filterwarnings("ignore")

In [2]:
def read_yaml_file(path, file):
    with open(os.path.join(path, file)) as f:
        try:
            content = yaml.safe_load(f)
        except yaml.YAMLError as e:
            raise e
    
    return content


CONFIG_PATH = os.path.join("..", "src", "config")

In [3]:
credentials = read_yaml_file(path=CONFIG_PATH, file="credentials.yaml")
settings = read_yaml_file(path=CONFIG_PATH, file="settings.yaml")

AWS_ACCESS_KEY = credentials['AWS_ACCESS_KEY']
AWS_SECRET_KEY = credentials['AWS_SECRET_KEY']
S3_NAME = credentials['S3']

ARTIFACTS_OUTPUT_PATH = settings['ARTIFACTS_PATH']
FEATURES_OUTPUT_PATH = settings['FEATURES_PATH']
RAW_FILE_PATH = os.path.join(settings["DATA_PATH"], settings["RAW_FILE_NAME"])
PROCESSED_RAW_FILE = "Preprocessed_" + settings["RAW_FILE_NAME"]
PROCESSED_RAW_FILE_PATH = os.path.join(settings["DATA_PATH"], PROCESSED_RAW_FILE)

In [4]:
settings["RAW_FILE_NAME"]

'Original_ObesityDataSet.csv'

In [5]:
RAW_FILE_PATH = f"../{RAW_FILE_PATH}"
PROCESSED_RAW_FILE_PATH = f"../{PROCESSED_RAW_FILE_PATH}"
ARTIFACTS_OUTPUT_PATH = f"../{ARTIFACTS_OUTPUT_PATH}"
FEATURES_OUTPUT_PATH = f"../{FEATURES_OUTPUT_PATH}"

In [6]:
# Inittials S3 client (for low-level operations)
s3_client = boto3.client(
    service_name = 's3',
    aws_access_key_id = AWS_ACCESS_KEY,
    aws_secret_access_key = AWS_SECRET_KEY
)
if not os.path.exists(RAW_FILE_PATH):
    s3_client.download_file(S3_NAME, settings["RAW_FILE_NAME"], RAW_FILE_PATH)

## Data cleaning

In [7]:
df = pd.read_csv(RAW_FILE_PATH)
df.drop('id', axis=1, inplace=True)
df.head()

,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,Male,24.443011,1.699998,81.669950,yes,yes,2.000000,2.983297,Sometimes,no,2.763573,no,0.000000,0.976473,Sometimes,Public_Transportation,Overweight_Level_II
1,Female,18.000000,1.560000,57.000000,yes,yes,2.000000,3.000000,Frequently,no,2.000000,no,1.000000,1.000000,no,Automobile,Normal_Weight
2,Female,18.000000,1.711460,50.165754,yes,yes,1.880534,1.411685,Sometimes,no,1.910378,no,0.866045,1.673584,no,Public_Transportation,Insufficient_Weight
3,Female,20.952737,1.710730,131.274851,yes,yes,3.000000,3.000000,Sometimes,no,1.674061,no,1.467863,0.780199,Sometimes,Public_Transportation,Obesity_Type_III
4,Male,31.641081,1.914186,93.798055,yes,yes,2.679664,1.971472,Sometimes,no,1.979848,no,1.967973,0.931721,Sometimes,Public_Transportation,Overweight_Level_II


### Removing Duplicates

In [8]:
df = df.drop_duplicates(keep='first')
pprint(f"Data Shape: {df.shape}")

'Data Shape: (20758, 17)'


### Transform Height units to Cetimeters

In [9]:
df['Height'] *= 100

### Removing Outliers

In [10]:
df.describe()

,Age,Height,Weight,FCVC,NCP,CH2O,FAF,TUE
count,20758.000000,20758.000000,20758.000000,20758.000000,20758.000000,20758.000000,20758.000000,20758.000000
mean,23.841804,170.024494,87.887768,2.445908,2.761332,2.029418,0.981747,0.616756
std,5.688072,8.731191,26.379443,0.533218,0.705375,0.608467,0.838302,0.602113
min,14.000000,145.000000,39.000000,1.000000,1.000000,1.000000,0.000000,0.000000
25%,20.000000,163.185600,66.000000,2.000000,3.000000,1.792022,0.008013,0.000000
50%,22.815416,170.000000,84.064875,2.393837,3.000000,2.000000,1.000000,0.573887
75%,26.000000,176.288700,111.600553,3.000000,3.000000,2.549617,1.587406,1.000000
max,61.000000,197.566300,165.057269,3.000000,4.000000,3.000000,3.000000,2.000000


In [11]:
# calculating the upper and lower limits
Q1 = df["Age"].quantile(0.25)
Q3 = df["Age"].quantile(0.75)
# threshold = 1.5
threshold = 3.0
IQR = Q3 - Q1

pprint(f"Dataset shape before removing the outliers: {df.shape}")

# removing the data samples that exceeds the upper or lower limits
df = df[~((df["Age"] >= (Q3 + threshold * IQR)) | (df["Age"] <= (Q1 - threshold * IQR)))]
pprint(f"Dataset shape after removing the outliers: {df.shape}")

'Dataset shape before removing the outliers: (20758, 17)'
'Dataset shape after removing the outliers: (20647, 17)'


## Creating New Features

### Body Mass Index (BMI)

In [12]:
df["BMI"] = df["Weight"] / (df["Height"] ** 2)

### Ideal Number of Main Meals? (INMM)

In [28]:
df["INMM"] = df["NCP"] == 3
df["INMM"] = df["INMM"].astype(int)

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 20647 entries, 0 to 20757
Data columns (total 19 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Gender                          20647 non-null  object 
 1   Age                             20647 non-null  float64
 2   Height                          20647 non-null  float64
 3   Weight                          20647 non-null  float64
 4   family_history_with_overweight  20647 non-null  object 
 5   FAVC                            20647 non-null  object 
 6   FCVC                            20647 non-null  float64
 7   NCP                             20647 non-null  float64
 8   CAEC                            20647 non-null  object 
 9   SMOKE                           20647 non-null  object 
 10  CH2O                            20647 non-null  float64
 11  SCC                             20647 non-null  object 
 12  FAF                             20647

### Transforming `Age` Column Into a Categorical Column

Reducing the impact of outliers in the `Age` column using *Quantile Bucketing*

In [15]:
values, bins = pd.qcut(x=df["Age"], q=4, retbins=True, labels=["q1", "q2", "q3", "q4"])

In [16]:
print(type(bins))
print(bins)

<class 'numpy.ndarray'>
[14.       20.       22.771001 26.       43.726081]


In [17]:

bins = np.concatenate(([-np.inf], bins[1:-1], [np.inf]))

df["Age"] = values
df["Age"] = df["Age"].astype("object")
df.head()

,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad,BMI,INMM
0,Male,q3,169.9998,81.669950,yes,yes,2.000000,2.983297,Sometimes,no,2.763573,no,0.000000,0.976473,Sometimes,Public_Transportation,Overweight_Level_II,0.002826,0
1,Female,q1,156.0000,57.000000,yes,yes,2.000000,3.000000,Frequently,no,2.000000,no,1.000000,1.000000,no,Automobile,Normal_Weight,0.002342,1
2,Female,q1,171.1460,50.165754,yes,yes,1.880534,1.411685,Sometimes,no,1.910378,no,0.866045,1.673584,no,Public_Transportation,Insufficient_Weight,0.001713,0
3,Female,q2,171.0730,131.274851,yes,yes,3.000000,3.000000,Sometimes,no,1.674061,no,1.467863,0.780199,Sometimes,Public_Transportation,Obesity_Type_III,0.004486,1
4,Male,q4,191.4186,93.798055,yes,yes,2.679664,1.971472,Sometimes,no,1.979848,no,1.967973,0.931721,Sometimes,Public_Transportation,Overweight_Level_II,0.002560,0


In [18]:
print(type(bins))
print(bins)

<class 'numpy.ndarray'>
[     -inf 20.       22.771001 26.             inf]


### Transforming `INMM` into Categorical Columns


In [19]:
df["INMM"] = df["INMM"].astype("object")
df.head()

,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad,BMI,INMM
0,Male,q3,169.9998,81.669950,yes,yes,2.000000,2.983297,Sometimes,no,2.763573,no,0.000000,0.976473,Sometimes,Public_Transportation,Overweight_Level_II,0.002826,0
1,Female,q1,156.0000,57.000000,yes,yes,2.000000,3.000000,Frequently,no,2.000000,no,1.000000,1.000000,no,Automobile,Normal_Weight,0.002342,1
2,Female,q1,171.1460,50.165754,yes,yes,1.880534,1.411685,Sometimes,no,1.910378,no,0.866045,1.673584,no,Public_Transportation,Insufficient_Weight,0.001713,0
3,Female,q2,171.0730,131.274851,yes,yes,3.000000,3.000000,Sometimes,no,1.674061,no,1.467863,0.780199,Sometimes,Public_Transportation,Obesity_Type_III,0.004486,1
4,Male,q4,191.4186,93.798055,yes,yes,2.679664,1.971472,Sometimes,no,1.979848,no,1.967973,0.931721,Sometimes,Public_Transportation,Overweight_Level_II,0.002560,0


### Spliting data into training and validation sets

In [20]:
X = df.drop("NObeyesdad", axis=1)
y = df["NObeyesdad"].values

In [21]:
X_train, X_val, y_train, y_val = train_test_split(X, y, train_size=0.8, stratify=y ,random_state=42)

X_train = X_train.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)

pprint(f"Train set shape: {X_train.shape} and {y_train.shape}")
pprint(f"Validation set shape: {X_val.shape} and {y_val.shape}")

'Train set shape: (16517, 18) and (16517,)'
'Validation set shape: (4130, 18) and (4130,)'


### Transform numerical columns (Log + 1 tranformation)

In [23]:
numerical_columns = df.select_dtypes(include=["number"]).columns.to_list()

for col in numerical_columns:
    X_train[col] = np.log1p(X_train[col])
    X_val[col] = np.log1p(X_val[col])

In [24]:
numerical_columns

['Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE', 'BMI']

### Scaling the numerical columns

In [ ]:
pprint("Training set skewness before scaling:")
pprint(X_train[numerical_columns].skew())
pprint("Validation set skewness before scaling:")
pprint(X_val[numerical_columns].skew())

'Training set skewness before scaling:'
Height    0.019782
Weight    0.086923
FCVC     -0.355868
NCP      -1.556255
CH2O     -0.214755
FAF       0.503176
TUE       0.672417
BMI       0.054401
dtype: float64
'Validation set skewness before scaling:'
Height   -0.008064
Weight    0.098152
FCVC     -0.373825
NCP      -1.570227
CH2O     -0.212550
FAF       0.500924
TUE       0.640379
BMI       0.051963
dtype: float64


**Fit:** Use `fit()` on your training data to calculate the mean and standard deviation for each feature. <br>
**Transform:** Use `transform()` to apply the scaling to your training and test data. It's crucial to use the same fitted scaler for both to ensure consistency.


- *Outliers:* `StandardScaler` can be sensitive to outliers
- *Data leakage:* Always fit the `StandardScaler` only on the training data and then apply the learned transformation to both training and test sets.

In [ ]:
scalers = {}

for col in numerical_columns:
    sc = StandardScaler()

    sc.fit(X_train[col].to_numpy().reshape(-1,1))

    X_train[col] = sc.transform(X_train[col].to_numpy().reshape(-1,1))
    X_val[col] = sc.transform(X_val[col].to_numpy().reshape(-1,1))
    scalers[col] = sc

In [ ]:
scalers

{'Height': StandardScaler(),
 'Weight': StandardScaler(),
 'FCVC': StandardScaler(),
 'NCP': StandardScaler(),
 'CH2O': StandardScaler(),
 'FAF': StandardScaler(),
 'TUE': StandardScaler(),
 'BMI': StandardScaler()}

In [ ]:
pprint("Training set skewness after scaling:")
pprint(X_train[numerical_columns].skew())
print()
pprint("Validation set skewness after scaling:")
pprint(X_val[numerical_columns].skew())

'Training set skewness after scaling:'
Height    0.019782
Weight    0.086923
FCVC     -0.355868
NCP      -1.556255
CH2O     -0.214755
FAF       0.503176
TUE       0.672417
BMI       0.054401
dtype: float64

'Validation set skewness after scaling:'
Height   -0.008064
Weight    0.098152
FCVC     -0.373825
NCP      -1.570227
CH2O     -0.212550
FAF       0.500924
TUE       0.640379
BMI       0.051963
dtype: float64


### Encoding categorical columns

In [ ]:
categorical_columns = X_train.select_dtypes(include=['object', 'category']).columns.to_list()

encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='infrequent_if_exist', min_frequency=20)
encoder.fit(X_train[categorical_columns])

train_encoder_df  = pd.DataFrame(
    data=encoder.transform(X_train[categorical_columns]),
    columns=encoder.get_feature_names_out(categorical_columns)
)

val_encoder_df  = pd.DataFrame(
    data=encoder.transform(X_val[categorical_columns]),
    columns=encoder.get_feature_names_out(categorical_columns)
)

new_train_df = pd.concat([train_encoder_df, X_train.drop(categorical_columns, axis=1)], axis=1)
new_val_df = pd.concat([val_encoder_df, X_val.drop(categorical_columns, axis=1)], axis=1)

X_train = new_train_df.values.copy()
X_val = new_val_df.values.copy()

In [ ]:
encoder.get_feature_names_out()

array(['Gender_Male', 'Age_q2', 'Age_q3', 'Age_q4',
       'family_history_with_overweight_yes', 'FAVC_yes',
       'CAEC_Frequently', 'CAEC_Sometimes', 'CAEC_no', 'SMOKE_yes',
       'SCC_yes', 'CALC_Sometimes', 'CALC_no', 'MTRANS_Bike',
       'MTRANS_Motorbike', 'MTRANS_Public_Transportation',
       'MTRANS_Walking', 'INMM_1'], dtype=object)

### Encoding the labels

In [ ]:
label_encoder = LabelBinarizer(sparse_output=False)
label_encoder.fit(y_train)

original_y_train = y_train.copy()
original_y_valid = y_val.copy()

y_train = label_encoder.transform(y_train)
y_val = label_encoder.transform(y_val)

In [ ]:
pprint(f"Train set shape: {X_train.shape} and {y_train.shape}")
pprint(f"Validation set shape: {X_val.shape} and {y_val.shape}")

'Train set shape: (16517, 26) and (16517, 7)'
'Validation set shape: (4130, 26) and (4130, 7)'


In [ ]:
label_encoder.classes_

array(['Insufficient_Weight', 'Normal_Weight', 'Obesity_Type_I',
       'Obesity_Type_II', 'Obesity_Type_III', 'Overweight_Level_I',
       'Overweight_Level_II'], dtype='<U19')

### Saving the Artifacts

In [ ]:
# saving the artifacts locally
os.makedirs(ARTIFACTS_OUTPUT_PATH, exist_ok=True)
os.makedirs(FEATURES_OUTPUT_PATH, exist_ok=True)

with open(os.path.join(ARTIFACTS_OUTPUT_PATH, 'scalers.pkl'), 'wb') as f:
    pickle.dump(scalers, f)
with open(os.path.join(ARTIFACTS_OUTPUT_PATH, 'features_encoder.pkl'), 'wb') as f:
    pickle.dump(encoder, f)
with open(os.path.join(ARTIFACTS_OUTPUT_PATH, 'label_encoder.pkl'), 'wb') as f:
    pickle.dump(label_encoder, f)
with open(os.path.join(ARTIFACTS_OUTPUT_PATH, 'qcut_bins.pkl'), 'wb') as f:
    pickle.dump(bins, f)


with open(os.path.join(FEATURES_OUTPUT_PATH, 'X_train.pkl'), 'wb') as f:
    pickle.dump(X_train, f)
with open(os.path.join(FEATURES_OUTPUT_PATH, 'X_val.pkl'), 'wb') as f:
    pickle.dump(X_val, f)
with open(os.path.join(FEATURES_OUTPUT_PATH, 'y_train.pkl'), 'wb') as f:
    pickle.dump(y_train, f)
with open(os.path.join(FEATURES_OUTPUT_PATH, 'y_val.pkl'), 'wb') as f:
    pickle.dump(y_val, f)

In [ ]:
# saving the preprocessed dataset locally
new_train_df['NObeyesdad'] = original_y_train
new_val_df['NObeyesdad'] = original_y_valid

preprocessed_data = pd.concat([new_train_df, new_val_df])
preprocessed_data.to_csv(PROCESSED_RAW_FILE_PATH, index=False, sep=",")

In [ ]:
def upload_folder_s3(root_path: str, s3_folder_prefix=""):
    try:
        for root, dirs, files in os.walk(root_path):
            for file_name in files:
                local_path = os.path.join(root, file_name)
                relative_path = os.path.relpath(local_path, root_path)

                if s3_folder_prefix:
                    s3_key = f"{s3_folder_prefix}/{relative_path}".replace("\\", "/")
                else:
                    s3_key = relative_path.replace("\\", "/")

                s3_client.upload_file(local_path, S3_NAME, s3_key)
                print(f"✅ Uploaded: {local_path} → s3://{S3_NAME}/{s3_key}")
    except Exception as err:
        print(f"❌ Upload failed: {err}")

if os.path.exists(ARTIFACTS_OUTPUT_PATH):
    upload_folder_s3(ARTIFACTS_OUTPUT_PATH, s3_folder_prefix="artifacts")

if os.path.exists(FEATURES_OUTPUT_PATH):
    upload_folder_s3(FEATURES_OUTPUT_PATH, s3_folder_prefix="features")

# sending preprocessed dataset saved locally to the aws s3 bucket
s3_client.upload_file(
    PROCESSED_RAW_FILE_PATH,
    credentials["S3"],
    PROCESSED_RAW_FILE
)

✅ Uploaded: ..//models/artifacts\features_encoder.pkl → s3://bucket6502-aws/artifacts/features_encoder.pkl
✅ Uploaded: ..//models/artifacts\label_encoder.pkl → s3://bucket6502-aws/artifacts/label_encoder.pkl
✅ Uploaded: ..//models/artifacts\qcut_bins.pkl → s3://bucket6502-aws/artifacts/qcut_bins.pkl
✅ Uploaded: ..//models/artifacts\scalers.pkl → s3://bucket6502-aws/artifacts/scalers.pkl
✅ Uploaded: ..//models/features\X_train.pkl → s3://bucket6502-aws/features/X_train.pkl
✅ Uploaded: ..//models/features\X_val.pkl → s3://bucket6502-aws/features/X_val.pkl


In [ ]:
# if os.path.exists(ARTIFACTS_OUTPUT_PATH):
#     shutil.rmtree(ARTIFACTS_OUTPUT_PATH)

# if os.path.exists(FEATURES_OUTPUT_PATH):
#     shutil.rmtree(FEATURES_OUTPUT_PATH)

# if os.path.exists(RAW_FILE_PATH):
#     os.remove(RAW_FILE_PATH)

# if os.path.exists(PROCESSED_RAW_FILE_PATH):
#     os.remove(PROCESSED_RAW_FILE_PATH)